In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
from functools import lru_cache

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.esic_v1 import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict
from sj_utils.evaluator import TimeChecker

In [ ]:
from rt_whisper import saveloaders
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/dev"
STORAGE = "/workspaces/dev/storage/esic/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
LOG_FILE_PATH = "/workspaces/dev/logs/core.log"

In [ ]:
src = Path(SOURCE)
storage = Path(STORAGE)
storage.mkdir(parents=True, exist_ok=True)
log = Path(LOG_FILE_PATH)
if log.exists():
    with log.open("w"): pass

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
@lru_cache(maxsize=128)
def load_mp4(mp4, sr=SAMPLE_RATE):
    return load_audio_from_mp4(mp4, sr=sr)

In [ ]:
def normalize_text(text):
    return normalize_text_only_en(text).upper()

In [ ]:
def token_saver(mp4: Path, save_path: Path) -> str:
    token_streamer = saveloaders.get_token_streamer_saver(
        save_path=save_path
    )
    audio, _ = load_mp4(mp4, sr=SAMPLE_RATE)

    completed = []
    param = Param()
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        transcribe_time.start()
        result:Result = token_streamer.process(param)
        transcribe_time.check()
        completed.extend(result.completed)
        param.update(result, update_prompt=True)
    completed.extend(result.candidate)

    text = " ".join([s.text for s in completed])
    text = normalize_text(text)

    return text

def token_loader(saved_path: Path) -> str:
    token_streamer = saveloaders.get_token_streamer_loader(
        saved_path=saved_path
    )

    segment_length = len(list(saved_path.iterdir()))

    completed = []
    param = Param()
    for _ in range(segment_length):
        param.language="en"
        transcribe_time.start()
        result:Result = token_streamer.process(param)
        transcribe_time.check()
        completed.extend(result.completed)
        param.update(result, update_prompt=False)
    completed.extend(result.candidate)

    text = " ".join([s.text for s in completed])
    text = normalize_text(text)

    return text

def transcriber(mp4: Path) -> TRNFormat:
    relative_path = mp4.parent.relative_to(src)
    saved_path = storage / relative_path

    if saved_path.exists():
        return token_loader(saved_path)
    return token_saver(mp4, saved_path)

In [ ]:
processed_time.start()
data = search_all_ref_and_hyp(src, transcriber, normalize_text, 1)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].append(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}